In [3]:
import pandas as pd
import random
from IPython.display import display, Markdown

from google.colab import files
uploaded = files.upload()

df = pd.read_excel("mcdonalds_menu_besin_degerleri.xlsx")

df.head()

Saving mcdonalds_menu_besin_degerleri.xlsx to mcdonalds_menu_besin_degerleri.xlsx


,Kategori,Ürün,Tüketim Birimi,Enerji (kcal/kj),Yağ (g),Doymuş Yağ (g),Trans Yağ (g),Karbonhidrat (g),Şeker (g),Protein (g),Tuz (g),Lif (g),Sodyum (mg),Kolesterol (mg)
0,SANDVİÇLER,Hamburger,1 porsiyon,"241,2 / 1008,5","7,7","4,4","0,0","30,4","5,1","13,5","1,3","2,2","503,3","6,6"
1,SANDVİÇLER,Hamburger,100 g,"247,7 / 1035,4","7,9","4,5","0,0","31,2","5,2","13,9","1,3","2,3","516,7","6,7"
2,SANDVİÇLER,Cheeseburger,1 porsiyon,"287,2 / 1200,3","11,3","7,8","0,0","30,9","5,1","16,3","1,7","2,2","680,2","9,1"
3,SANDVİÇLER,Cheeseburger,100 g,"258,5 / 1080,4","10,2","7,0","0,0","27,8","4,6","14,7","1,6","2,0","612,3","8,1"
4,SANDVİÇLER,Double Cheeseburger,1 porsiyon,"422,7 / 1766,9","20,8","14,5","0,1","32,0","5,1","28,0","2,7","2,4","1082,3","18,0"


In [6]:
df["Kategori"] = df["Kategori"].replace({
    "KAHVALTILIKLAR (Devam)": "KAHVALTILIKLAR",
    "İÇECEKLER (Smoothie/Meyve Suyu)": "İÇECEKLER",
})

sorted(df["Kategori"].unique())

['ATIŞTIRMALIKLAR',
 'HAPPY MEAL MENÜLERİ',
 'KAHVALTILIKLAR',
 'MILKSHAKE VE TATLILAR',
 'McD CAFÉ',
 'SALATALAR',
 'SANDVİÇLER',
 'SOSLAR',
 'İÇECEKLER']

In [9]:
# Enerji sütunundan sadece kcal değerini al
if df["Enerji (kcal/kj)"].dtype == object:
    df["Kalori"] = (
        df["Enerji (kcal/kj)"]
        .str.split("/").str[0].str.strip()
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

# Sayısal hale getirilecek sütunlar
numeric_columns = [
    "Yağ (g)", "Doymuş Yağ (g)", "Trans Yağ (g)", "Karbonhidrat (g)",
    "Şeker (g)", "Protein (g)", "Tuz (g)", "Lif (g)", "Sodyum (mg)", "Kolesterol (mg)"
]
for col in numeric_columns:
    if df[col].dtype == object:  # sadece hâlâ metinse dönüştür
        df[col] = df[col].str.replace(",", ".", regex=False).astype(float)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255 entries, 0 to 254
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Kategori          255 non-null    object 
 1   Ürün              255 non-null    object 
 2   Tüketim Birimi    255 non-null    object 
 3   Enerji (kcal/kj)  255 non-null    object 
 4   Yağ (g)           255 non-null    float64
 5   Doymuş Yağ (g)    255 non-null    float64
 6   Trans Yağ (g)     255 non-null    float64
 7   Karbonhidrat (g)  255 non-null    float64
 8   Şeker (g)         255 non-null    float64
 9   Protein (g)       255 non-null    float64
 10  Tuz (g)           255 non-null    float64
 11  Lif (g)           255 non-null    float64
 12  Sodyum (mg)       255 non-null    float64
 13  Kolesterol (mg)   255 non-null    float64
 14  Kalori            255 non-null    float64
 15  _ml               102 non-null    float64
 16  _is_100g          255 non-null    bool   
 1

In [10]:
# 100 g satırı farklı bir isimle kayıtlı olan ürünler için eşleşme sözlüğü
REF_ESLESME = {
    "Chicken McNuggets 4'lü": "Chicken McNuggets",
    "Chicken McNuggets 6'lı": "Chicken McNuggets",
    "Çıtır Soğan 10'lu": "Çıtır Soğan",
    "Patates kızartması - Küçük Boy": "Patates Kızartması",
    "Patates Kızartması - Orta Boy": "Patates Kızartması",
    "Patates kızartması - Büyük Boy": "Patates Kızartması",
    "Patates Kızartması - Süper Boy": "Patates Kızartması",
}

# ml miktarını Tüketim Birimi'nden ayıkla (örn. "250 ml" -> 250)
df["_ml"] = df["Tüketim Birimi"].str.extract(r"(\d+)\s*ml").astype(float)
df["_is_100g"] = df["Tüketim Birimi"] == "100 g"
df["_is_100ml"] = df["_ml"] == 100
df["_is_porsiyon"] = df["Tüketim Birimi"] == "1 porsiyon"

records = []
for (kategori, urun), g in df.groupby(["Kategori", "Ürün"], sort=False):
    is_drink = g["_ml"].notna().any()

    if is_drink:
        # İçecekler: yoğunluk referansı 100 ml, gösterilecek porsiyon ise
        # 100 ml'den büyük en küçük hacim (örn. Coca-Cola'da 250 ml)
        ref_row = g[g["_is_100ml"]]
        aday = g[g["_ml"] > 100]
        serve_row = aday.loc[[aday["_ml"].idxmin()]] if not aday.empty else g[g["_ml"].notna()].loc[[g["_ml"].idxmin()]]
    else:
        ref_row = g[g["_is_100g"]]
        serve_row = g[g["_is_porsiyon"]]
        if serve_row.empty:
            # "1 porsiyon" hiç yoksa (örn. Beyaz Peynirli-Kepekli Tost),
            # 100 g satırının kendisi porsiyon kabul edilir
            serve_row = ref_row
        if ref_row.empty and urun in REF_ESLESME:
            eslesen_urun = REF_ESLESME[urun]
            ref_row = df[(df["Kategori"] == kategori) & (df["Ürün"] == eslesen_urun) & (df["_is_100g"])]

    if ref_row.empty or serve_row.empty:
        continue

    ref, serve = ref_row.iloc[0], serve_row.iloc[0]
    porsiyon_g = round(serve["Kalori"] / ref["Kalori"] * 100, 1) if ref["Kalori"] > 0 else 100.0

    records.append({
        "Kategori": kategori,
        "Ürün": urun,
        "Tüketim Birimi": serve["Tüketim Birimi"],
        "Kalori": serve["Kalori"],
        "Porsiyon (g)": porsiyon_g,
        "Protein (g)": serve["Protein (g)"],
        "Şeker (g)": serve["Şeker (g)"],
    })

df_serving = pd.DataFrame(records)
df_serving["Protein (%)"] = (df_serving["Protein (g)"] / df_serving["Porsiyon (g)"] * 100).round(1)
df_serving["Şeker (%)"] = (df_serving["Şeker (g)"] / df_serving["Porsiyon (g)"] * 100).round(1)

print("Toplam ürün:", df["Ürün"].nunique(), "| df_serving satır sayısı:", len(df_serving))
df_serving.head()

Toplam ürün: 113 | df_serving satır sayısı: 113


,Kategori,Ürün,Tüketim Birimi,Kalori,Porsiyon (g),Protein (g),Şeker (g),Protein (%),Şeker (%)
0,SANDVİÇLER,Hamburger,1 porsiyon,241.2,97.4,13.5,5.1,13.9,5.2
1,SANDVİÇLER,Cheeseburger,1 porsiyon,287.2,111.1,16.3,5.1,14.7,4.6
2,SANDVİÇLER,Double Cheeseburger,1 porsiyon,422.7,158.4,28.0,5.1,17.7,3.2
3,SANDVİÇLER,Big Mac,1 porsiyon,492.9,217.5,27.8,7.1,12.8,3.3
4,SANDVİÇLER,Double Big Mac,1 porsiyon,672.2,284.7,45.5,7.1,16.0,2.5


In [11]:
top10_calories = df_serving.nlargest(10, "Kalori")[["Ürün", "Kategori", "Kalori"]]
top10_calories

,Ürün,Kategori,Kalori
6,Double Quarter Pounder,SANDVİÇLER,764.80
40,Türk Kahvaltı Tabağı,KAHVALTILIKLAR,744.60
4,Double Big Mac,SANDVİÇLER,672.20
73,Doğum Günü Pastası,MILKSHAKE VE TATLILAR,596.00
18,Showburger Etli Karabiber Soslu,SANDVİÇLER,583.50
93,Çikolatalı Muffin,McD CAFÉ,557.18
17,Showburger Etli Barbekü Soslu,SANDVİÇLER,550.00
5,Quarter Pounder,SANDVİÇLER,539.60
14,Double Acılı Tavuk,SANDVİÇLER,532.60
20,Showburger Tavuklu Karabiber Soslu,SANDVİÇLER,521.40


In [12]:
for category in df_serving["Kategori"].unique():
    display(Markdown(f"## {category}"))

    table = (
        df_serving[df_serving["Kategori"] == category]
        .nlargest(5, "Kalori")[["Ürün", "Kalori"]]
        .reset_index(drop=True)
    )
    display(table)

## SANDVİÇLER

,Ürün,Kalori
0,Double Quarter Pounder,764.8
1,Double Big Mac,672.2
2,Showburger Etli Karabiber Soslu,583.5
3,Showburger Etli Barbekü Soslu,550.0
4,Quarter Pounder,539.6


## ATIŞTIRMALIKLAR

,Ürün,Kalori
0,Patates Kızartması - Süper Boy,511.8
1,Patates kızartması - Büyük Boy,434.9
2,Çıtır Soğan 10'lu,383.0
3,Çıtır Soğan,354.6
4,Patates Kızartması,319.9


## SALATALAR

...

,Ürün,Kalori
0,Doğum Günü Pastası,596.0
1,Cool Cake,403.3
2,Çikolatalı Donut,318.0
3,Karamel Soslu Sundae,284.9
4,Tarçınlı Donut,271.9


## SOSLAR

,Ürün,Kalori
0,Zeytinyağlı Limonlu Sos,178.62
1,Ranch Sos,75.60
2,Buffalo Sos,54.40
3,Acı Hardal,45.10
4,Sarımsaklı Mayonez,43.50


## McD CAFÉ

,Ürün,Kalori
0,Çikolatalı Muffin,557.18
1,Havuçlu Kek,485.04
2,Mozaik Pasta,349.00
3,Limonlu Cheesecake,331.40
4,Frambuazlı Cheesecake,322.30


In [13]:
shown_products = set()

for category in df_serving["Kategori"].unique():
    display(Markdown(f"## {category}"))

    temp = df_serving[df_serving["Kategori"] == category].sort_values("Kalori")
    temp = temp[~temp["Ürün"].isin(shown_products)]
    temp = temp.head(5)[["Ürün", "Kalori"]].reset_index(drop=True)

    display(temp)
    shown_products.update(temp["Ürün"])

## SANDVİÇLER

,Ürün,Kalori
0,Hamburger,241.2
1,Cheeseburger,287.2
2,McChicken,360.5
3,Çıtır Tavuk,377.0
4,Acılı Tavuk,379.8


## ATIŞTIRMALIKLAR

,Ürün,Kalori
0,Chicken McNuggets 4'lü,147.1
1,Chicken McNuggets 6'lı,220.6
2,Chicken McNuggets,233.9
3,Patates kızartması - Küçük Boy,240.3
4,Acılı McBites,267.3


## SALATALAR

...

,Ürün,Kalori
0,Külah,109.7
1,Kornet,118.7
2,Böğürtlen Soslu Sundae,190.2
3,Hindistancevizli Milkshake,249.5
4,Çilekli Milkshake,252.5


## SOSLAR

,Ürün,Kalori
0,Acı Sos,4.8
1,Ketçap,12.8
2,Barbekü Sos,30.7
3,Mayonez,34.6
4,Sarımsaklı Mayonez,43.5


## McD CAFÉ

,Ürün,Kalori
0,Amerikano,10.20
1,Espresso,17.60
2,Türk Kahvesi,18.96
3,Espresso Machiato,20.16
4,Mini Ekler,62.40


In [15]:
for category in df_serving["Kategori"].unique():
    display(Markdown(f"## {category}"))

    table = (
        df_serving[df_serving["Kategori"] == category]
        .sort_values("Protein (%)", ascending=False)[
            ["Ürün", "Porsiyon (g)", "Protein (g)", "Protein (%)"]
        ]
        .rename(columns={"Protein (g)": "1 Porsiyondaki Protein (g)"})
        .reset_index(drop=True)
    )
    display(table)

    display(Markdown(
        "**Not:** Protein (%) değeri, ürünün toplam porsiyon ağırlığının yüzde kaçının proteinden oluştuğunu göstermektedir."
    ))

## SANDVİÇLER

,Ürün,Porsiyon (g),1 Porsiyondaki Protein (g),Protein (%)
0,Double Quarter Pounder,291.8,58.8,20.2
1,Double Cheeseburger,158.4,28.0,17.7
2,Quarter Pounder,203.5,35.6,17.5
3,Double Big Mac,284.7,45.5,16.0
4,Cheeseburger,111.1,16.3,14.7
5,McRoyal,238.1,33.4,14.0
6,Hamburger,97.4,13.5,13.9
7,Showburger Tavuklu Karabiber Soslu,214.7,29.8,13.9
8,Showburger Tavuklu Buffalo Soslu,215.0,29.7,13.8
9,Double Acılı Tavuk,301.1,40.9,13.6


**Not:** Protein (%) değeri, ürünün toplam porsiyon ağırlığının yüzde kaçının proteinden oluştuğunu göstermektedir.

## ATIŞTIRMALIKLAR

,Ürün,Porsiyon (g),1 Porsiyondaki Protein (g),Protein (%)
0,Chicken McNuggets 4'lü,62.9,12.9,20.5
1,Chicken McNuggets 6'lı,94.3,19.3,20.5
2,Chicken McNuggets,100.0,20.5,20.5
3,Acılı McBites,111.2,15.9,14.3
4,Çıtır Soğan 10'lu,108.0,5.4,5.0
5,Çıtır Soğan,100.0,5.0,5.0
6,Patates kızartması - Küçük Boy,75.1,3.2,4.3
7,Patates Kızartması - Orta Boy,96.9,4.1,4.2
8,Patates kızartması - Büyük Boy,135.9,5.7,4.2
9,Patates Kızartması - Süper Boy,160.0,6.7,4.2


...

,Ürün,Porsiyon (g),1 Porsiyondaki Protein (g),Protein (%)
0,Acı Sos,10.1,0.2,2.0
1,Ketçap,11.0,0.2,1.8
2,Acı Hardal,23.5,0.4,1.7
3,Ranch Sos,21.0,0.2,1.0
4,Mayonez,11.2,0.1,0.9
5,Sarımsaklı Mayonez,11.2,0.1,0.9
6,Buffalo Sos,21.0,0.1,0.5
7,Barbekü Sos,23.4,0.1,0.4
8,Zeytinyağlı Limonlu Sos,30.0,0.0,0.0


**Not:** Protein (%) değeri, ürünün toplam porsiyon ağırlığının yüzde kaçının proteinden oluştuğunu göstermektedir.

## McD CAFÉ

,Ürün,Porsiyon (g),1 Porsiyondaki Protein (g),Protein (%)
0,Makaron,20.0,1.6,8.0
1,Mini Ekler,22.0,1.6,7.3
2,Profiterollü Pasta,85.0,6.2,7.3
3,Frambuazlı Cheesecake,100.0,7.0,7.0
4,Çikolatalı Muffin,130.0,7.8,6.0
5,Tiramusu,100.0,5.7,5.7
6,Yabanmersinli Mini Muffin,30.0,1.6,5.3
7,Limonlu Cheesecake,100.0,4.8,4.8
8,Mozaik Pasta,95.0,4.5,4.7
9,Frambuazlı Pasta,80.0,2.2,2.8


**Not:** Protein (%) değeri, ürünün toplam porsiyon ağırlığının yüzde kaçının proteinden oluştuğunu göstermektedir.

In [16]:
for category in df_serving["Kategori"].unique():
    display(Markdown(f"## {category}"))

    table = (
        df_serving[df_serving["Kategori"] == category]
        .sort_values("Şeker (%)", ascending=False)[
            ["Ürün", "Porsiyon (g)", "Şeker (g)", "Şeker (%)"]
        ]
        .rename(columns={"Şeker (g)": "1 Porsiyondaki Şeker (g)"})
        .reset_index(drop=True)
    )
    display(table)

    display(Markdown(
        "**Not:** Şeker (%) değeri, ürünün toplam porsiyon ağırlığının yüzde kaçının şekerden oluştuğunu göstermektedir."
    ))

## SANDVİÇLER

,Ürün,Porsiyon (g),1 Porsiyondaki Şeker (g),Şeker (%)
0,Hamburger,97.4,5.1,5.2
1,Showburger Etli Barbekü Soslu,206.0,10.3,5.0
2,Double Köfteburger,191.5,9.5,5.0
3,Cheeseburger,111.1,5.1,4.6
4,McChicken,170.3,7.9,4.6
5,Acılı Tavuk,214.5,9.3,4.3
6,McJR,174.6,7.5,4.3
7,Köfteburger,156.1,6.5,4.2
8,McRoyal,238.1,8.7,3.7
9,Showburger Etli Karabiber Soslu,204.1,7.3,3.6


**Not:** Şeker (%) değeri, ürünün toplam porsiyon ağırlığının yüzde kaçının şekerden oluştuğunu göstermektedir.

## ATIŞTIRMALIKLAR

,Ürün,Porsiyon (g),1 Porsiyondaki Şeker (g),Şeker (%)
0,Çıtır Soğan 10'lu,108.0,2.7,2.5
1,Çıtır Soğan,100.0,2.5,2.5
2,Patates kızartması - Küçük Boy,75.1,0.5,0.7
3,Patates Kızartması - Süper Boy,160.0,1.1,0.7
4,Patates kızartması - Büyük Boy,135.9,1.0,0.7
5,Patates Kızartması,100.0,0.7,0.7
6,Patates Kızartması - Orta Boy,96.9,0.7,0.7
7,Acılı McBites,111.2,0.7,0.6
8,Chicken McNuggets,100.0,0.0,0.0
9,Chicken McNuggets 4'lü,62.9,0.0,0.0


...

,Ürün,Porsiyon (g),1 Porsiyondaki Şeker (g),Şeker (%)
0,Acı Hardal,23.5,6.9,29.4
1,Barbekü Sos,23.4,6.2,26.5
2,Ketçap,11.0,2.4,21.8
3,Buffalo Sos,21.0,0.9,4.3
4,Ranch Sos,21.0,0.8,3.8
5,Acı Sos,10.1,0.3,3.0
6,Sarımsaklı Mayonez,11.2,0.1,0.9
7,Mayonez,11.2,0.0,0.0
8,Zeytinyağlı Limonlu Sos,30.0,0.0,0.0


**Not:** Şeker (%) değeri, ürünün toplam porsiyon ağırlığının yüzde kaçının şekerden oluştuğunu göstermektedir.

## McD CAFÉ

,Ürün,Porsiyon (g),1 Porsiyondaki Şeker (g),Şeker (%)
0,Mozaik Pasta,95.0,52.3,55.1
1,Makaron,20.0,9.2,46.0
2,Havuçlu Kek,120.0,43.2,36.0
3,Çikolatalı Muffin,130.0,45.6,35.1
4,Limonlu Cheesecake,100.0,35.0,35.0
5,Yabanmersinli Mini Muffin,30.0,10.1,33.7
6,Frambuazlı Pasta,80.0,26.3,32.9
7,Frambuazlı Cheesecake,100.0,31.9,31.9
8,Profiterollü Pasta,85.0,26.5,31.2
9,Tiramusu,100.0,25.8,25.8


**Not:** Şeker (%) değeri, ürünün toplam porsiyon ağırlığının yüzde kaçının şekerden oluştuğunu göstermektedir.

In [21]:
import ipywidgets as widgets
from IPython.display import clear_output

kategori_secici = widgets.Dropdown(
    options=sorted(df_serving["Kategori"].unique()),
    description="Kategori:",
    style={"description_width": "initial"}
)

urun_secici = widgets.Dropdown(
    description="Ürün:",
    style={"description_width": "initial"}
)

buton = widgets.Button(description="Öneri Getir", button_style="success")
cikti = widgets.Output()

def kategoriyi_guncelle(change=None):
    secilen_kategori = kategori_secici.value
    urunler = sorted(
        df_serving[df_serving["Kategori"] == secilen_kategori]["Ürün"].tolist()
    )
    urun_secici.options = urunler

kategoriyi_guncelle()
kategori_secici.observe(kategoriyi_guncelle, names="value")

def oneri_getir(b):
    with cikti:
        clear_output()

        secilen = df_serving[
            (df_serving["Kategori"] == kategori_secici.value) &
            (df_serving["Ürün"] == urun_secici.value)
        ]

        if secilen.empty:
            print("❌ Ürün bulunamadı.")
            return

        secilen = secilen.iloc[0]
        kategori = secilen["Kategori"]
        kalori = secilen["Kalori"]
        protein = secilen["Protein (%)"]
        seker = secilen["Şeker (%)"]

        ayni_kategori = df_serving[
            (df_serving["Kategori"] == kategori) &
            (df_serving["Ürün"] != secilen["Ürün"])
        ]

        az_kalori = ayni_kategori[ayni_kategori["Kalori"] < kalori]
        cok_protein = ayni_kategori[ayni_kategori["Protein (%)"] > protein]
        az_seker = ayni_kategori[ayni_kategori["Şeker (%)"] < seker]

        print(f"Seçtiğiniz ürün: {secilen['Ürün']}")
        print(f"Kategori: {kategori}")

        print("\n🔥 Daha az kalorili öneri:")
        print(random.choice(az_kalori["Ürün"].tolist()) if not az_kalori.empty else "Uygun alternatif bulunamadı. Bu kategorideki en az kalorili ürün bu. Tebrikler!")

        print("\n💪 Daha yüksek protein oranına sahip öneri:")
        print(random.choice(cok_protein["Ürün"].tolist()) if not cok_protein.empty else "Uygun alternatif bulunamadı. Bu kategorideki en proteinli ürün bu. Tebrikler!")

        print("\n🍭 Daha düşük şeker oranına sahip öneri:")
        print(random.choice(az_seker["Ürün"].tolist()) if not az_seker.empty else "Uygun alternatif bulunamadı. Bu kategorideki en az şekerli ürün bu. Tebrikler!")

buton.on_click(oneri_getir)

display(kategori_secici, urun_secici, buton, cikti)

Dropdown(description='Kategori:', options=('ATIŞTIRMALIKLAR', 'HAPPY MEAL MENÜLERİ', 'KAHVALTILIKLAR', 'MILKSH…

Dropdown(description='Ürün:', options=('Acılı McBites', 'Chicken McNuggets', "Chicken McNuggets 4'lü", "Chicke…

Button(button_style='success', description='Öneri Getir', style=ButtonStyle())

Output()